# Video RAG — Phase 1: Notebook 01
## Video Audio Extraction and Timestamped Speech-to-Text Transcription

This notebook demonstrates the first two stages of the research pipeline:
1. **Phase 1A**: Validating the video file and extracting mono 16kHz audio using FFmpeg without modifying the original video.
2. **Phase 1B**: Transcribing the speech into timestamp-preserved segments using Whisper (`[{"start": float, "end": float, "text": str}]`).


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

# Add project root to sys.path
sys.path.append(str(Path.cwd().parent))

from src.video_processor import VideoProcessor
from src.transcription import Transcriber, format_timestamp


### Step 1: Video File Validation & Audio Extraction


In [ ]:
video_path = "../data/videos/sample.mp4"

processor = VideoProcessor(output_audio_dir="../data/audio")
metadata = processor.get_media_metadata(video_path)
print("Video Metadata:")
for k, v in metadata.items():
    print(f"  {k}: {v}")

# Extract audio stream (16kHz mono WAV suitable for Whisper)
audio_path = processor.extract_audio(video_path, output_format="wav", overwrite=True)
print(f"\nExtracted Audio Path: {audio_path}")


### Step 2: Timestamped Speech-to-Text Transcription with Whisper


In [ ]:
# Initialize Transcriber (supports faster-whisper and openai-whisper)
transcriber = Transcriber(model_size="base", output_dir="../data/transcripts")

# Transcribe the audio
segments = transcriber.transcribe(audio_path, language="en")

print(f"Total Speech Segments Extracted: {len(segments)}")


### Step 3: Inspect Timestamped Segments Table


In [ ]:
# Display segments in a clean pandas DataFrame
records = [
    {
        "Index": i + 1,
        "Start Time (s)": seg.start,
        "End Time (s)": seg.end,
        "Formatted Range": f"[{format_timestamp(seg.start)} - {format_timestamp(seg.end)}]",
        "Duration (s)": round(seg.duration, 2),
        "Text": seg.text
    }
    for i, seg in enumerate(segments)
]
df_segments = pd.DataFrame(records)
df_segments.head(10)


### Step 4: Save Structured JSON & Human-Readable TXT Transcripts


In [ ]:
saved_files = transcriber.save_transcripts(
    segments=segments,
    video_name="sample.mp4",
    metadata={"video_size_mb": metadata["size_mb"]}
)

print(f"Structured JSON: {saved_files['json']}")
print(f"Human-Readable TXT: {saved_files['txt']}")
